# Directors Tracking Dashboard

This notebook shows which files pulled directors successfully and tracks the director names by company and year.

In [1]:
import json
from pathlib import Path
import pandas as pd

workspace = Path('/Users/morganlin/Library/CloudStorage/OneDrive-SharedLibraries-VillanovaUniversity/Brian Grant - Grant and Lin/sample')
directors_file = workspace / 'directors_output_clean' / 'directors_all_clean.json'
metrics_file = workspace / 'pl_metrics_cleaned' / 'all_pl_metrics.json'

with open(directors_file, 'r') as f:
    directors_data = json.load(f)

with open(metrics_file, 'r') as f:
    metrics_data = json.load(f)

tracking_rows = []
for item in directors_data:
    source_file = item.get('file', '')
    base_name = source_file.replace('.pdf', '').replace(' account1', '').replace(' account', '')
    company_id = base_name.split('_')[0] if '_' in base_name else base_name.split('-')[0]
    year = ''
    for token in base_name.replace('-', '_').split('_'):
        if token.isdigit() and len(token) == 4:
            year = token
            break

    director_block = item.get('directors', {}) or {}
    directors = director_block.get('Director', [])
    if not isinstance(directors, list):
        directors = [str(directors)] if directors else []

    other_roles = []
    for role, names in director_block.items():
        if role != 'Director':
            if isinstance(names, list):
                other_roles.extend([f'{role}: {name}' for name in names])
            elif names:
                other_roles.append(f'{role}: {names}')

    tracking_rows.append({
        'company_id': company_id,
        'year': year,
        'source_file': source_file,
        'directors': directors,
        'other_roles': other_roles,
    })

metrics_df = pd.DataFrame(metrics_data)
metrics_df['company_id'] = metrics_df['filename'].apply(lambda x: str(x).split('_')[0] if '_' in str(x) else str(x).split('-')[0])
metrics_df['year'] = metrics_df['filename'].apply(lambda x: ''.join(ch for ch in str(x) if ch.isdigit())[:4] if any(ch.isdigit() for ch in str(x)) else '')

print(f'Loaded {len(tracking_rows)} director rows from {len(directors_data)} source files')
print(f'Loaded {len(metrics_df)} income statement rows from {len(metrics_data)} source files')
print('\nSample director rows:')
print(pd.DataFrame(tracking_rows)[['company_id', 'year', 'source_file']].head(10).to_string(index=False))

Loaded 20 director rows from 20 source files
Loaded 24 income statement rows from 24 source files

Sample director rows:
company_id year             source_file
    138763 2020 138763_2020 account.pdf
    138763 2021 138763_2021 account.pdf
    182294 2014 182294_2014 account.pdf
    182294 2018 182294_2018 account.pdf
    182294 2019 182294_2019 account.pdf
    182294 2020 182294_2020 account.pdf
    182294 2021 182294_2021 account.pdf
    182294 2022 182294_2022 account.pdf
     76927 2018  76927_2018 account.pdf
     76927 2019  76927_2019 account.pdf


## All Directors by File

In [2]:
tracking_df = pd.DataFrame(tracking_rows)
if 'year_num' not in tracking_df.columns:
    tracking_df['year_num'] = pd.to_numeric(tracking_df['year'], errors='coerce')
tracking_df['year_num'] = pd.to_numeric(tracking_df['year'], errors='coerce')
tracking_df['director_count'] = tracking_df['directors'].apply(lambda x: len(x) if isinstance(x, list) else 0)
tracking_df['has_directors'] = tracking_df['director_count'] > 0
tracking_df['other_role_count'] = tracking_df['other_roles'].apply(lambda x: len(x) if isinstance(x, list) else 0)
tracking_df['directors_text'] = tracking_df['directors'].apply(lambda x: ', '.join(x) if isinstance(x, list) and x else 'N/A')
tracking_df['other_roles_text'] = tracking_df['other_roles'].apply(lambda x: ', '.join(x) if isinstance(x, list) and x else 'N/A')

print(f"Loaded {len(tracking_df)} extracted director rows")
print(f"Companies detected: {tracking_df['company_id'].nunique()}")
print(f"Rows with directors: {tracking_df['has_directors'].sum()}")
tracking_df[['company_id', 'year', 'source_file', 'director_count', 'has_directors', 'directors_text', 'other_roles_text']].sort_values(['company_id', 'year_num', 'source_file'])

Loaded 20 extracted director rows
Companies detected: 6
Rows with directors: 8


KeyError: 'year_num'

## Directors Coverage Summary

In [3]:
print(f"Files with at least one director entry: {tracking_df['has_directors'].sum()}/{len(tracking_df)}")
print(f"Companies represented: {tracking_df['company_id'].nunique()}")
print(f"Rows with 2+ directors: {(tracking_df['director_count'] >= 2).sum()}")
print(f"Rows with 3+ directors: {(tracking_df['director_count'] >= 3).sum()}")
print("\nFiles without director entries:")
print(tracking_df.loc[~tracking_df['has_directors'], 'source_file'].head(25).to_string(index=False))

Files with at least one director entry: 8/20
Companies represented: 6
Rows with 2+ directors: 8
Rows with 3+ directors: 7

Files without director entries:
138763_2020 account.pdf
182294_2014 account.pdf
182294_2019 account.pdf
182294_2020 account.pdf
182294_2021 account.pdf
182294_2022 account.pdf
 76927_2018 account.pdf
 76927_2019 account.pdf
 76927_2021 account.pdf
 79963_2018 account.pdf
 79963_2019 account.pdf
904750_2024 account.pdf


## Income Statement Data

In [4]:
# Show every file row, not just the latest company row
print(f"All director rows: {len(tracking_df)}")
display_cols = ['company_id', 'year', 'source_file', 'director_count', 'has_directors', 'directors_text', 'other_roles_text']
tracking_df[display_cols].sort_values(['company_id', 'year_num', 'source_file']) if 'year_num' in tracking_df.columns else tracking_df[display_cols].sort_values(['company_id', 'year', 'source_file'])

All director rows: 20


KeyError: 'year_num'

In [5]:
income_df = metrics_df.copy()
income_df['turnover_text'] = income_df['turnover'].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else 'N/A')
income_df['operating_profit_text'] = income_df['operating_profit'].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else 'N/A')
income_df['profit_before_tax_text'] = income_df['profit_before_tax'].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else 'N/A')
income_df['profit_loss_for_year_text'] = income_df['profit_loss_for_year'].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else 'N/A')

print(f"Income statement rows: {len(income_df)}")
print(f"Rows with any metric: {income_df[['turnover', 'operating_profit', 'profit_before_tax', 'profit_loss_for_year']].notna().any(axis=1).sum()}")

income_df[['filename', 'company_id', 'year', 'pl_found', 'pl_start_line', 'turnover_text', 'operating_profit_text', 'profit_before_tax_text', 'profit_loss_for_year_text', 'currency']].sort_values(['company_id', 'year']) if 'year' in income_df.columns else income_df[['filename', 'company_id', 'pl_found', 'pl_start_line', 'turnover_text', 'operating_profit_text', 'profit_before_tax_text', 'profit_loss_for_year_text', 'currency']]

Income statement rows: 24
Rows with any metric: 13


,filename,company_id,year,pl_found,pl_start_line,turnover_text,operating_profit_text,profit_before_tax_text,profit_loss_for_year_text,currency
3,138763_2020 account_pl,138763,1387,False,-1,N/A,N/A,N/A,N/A,Unknown
4,138763_2021 account_pl,138763,1387,True,296,"31,442","2,086",N/A,"2,305",EUR
0,138763-Account_details_for_ARCROYAL_LIMITED-20...,138763-Account,1387,False,-1,N/A,N/A,N/A,N/A,Unknown
1,138763-Account_details_for_ARCROYAL_LIMITED-20...,138763-Account,1387,False,-1,N/A,N/A,N/A,N/A,Unknown
2,138763-Account_details_for_ARCROYAL_LIMITED-20...,138763-Account,1387,False,-1,N/A,N/A,N/A,N/A,Unknown
5,182294_2014 account_pl,182294,1822,False,-1,N/A,N/A,N/A,N/A,Unknown
6,182294_2018 account_pl,182294,1822,False,-1,N/A,N/A,N/A,N/A,Unknown
7,182294_2019 account_pl,182294,1822,False,-1,N/A,N/A,N/A,201,Unknown
8,182294_2020 account_pl,182294,1822,False,-1,N/A,N/A,N/A,N/A,Unknown
9,182294_2021 account_pl,182294,1822,False,-1,N/A,N/A,N/A,N/A,Unknown


## All Income Statement Rows

In [ ]:
# Optional: current year tracking by company, using a safe year sort
current_by_company = tracking_df.copy()
current_by_company = current_by_company.sort_values(['company_id', 'year_num', 'source_file']) if 'year_num' in current_by_company.columns else current_by_company.sort_values(['company_id', 'year', 'source_file'])
current_by_company = current_by_company.groupby('company_id', as_index=False).tail(1)
print(f"Current company rows: {len(current_by_company)}")
current_by_company[['company_id', 'year', 'source_file', 'director_count', 'has_directors', 'directors_text', 'other_roles_text']]